# Double Inverted Pendulum: Free Dynamics and LQR Stabilization

This notebook focuses on the local behavior around the upright equilibrium `(0, 0, 0, 0)`. It compares the uncontrolled system with an LQR stabilizer designed around that equilibrium.

## System Sketch

The sketch below shows the geometry used in the notebook. Both angles are measured from the upward vertical reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

theta1_demo = 0.55
theta2_demo = -0.35
l1_demo = 1.0
l2_demo = 0.85

pivot = np.array([0.0, 0.0])
joint1 = np.array([l1_demo * np.sin(theta1_demo), l1_demo * np.cos(theta1_demo)])
joint2 = joint1 + np.array([l2_demo * np.sin(theta2_demo), l2_demo * np.cos(theta2_demo)])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([pivot[0], joint1[0]], [pivot[1], joint1[1]], 'o-', lw=3, color='tab:blue', label='link 1')
ax.plot([joint1[0], joint2[0]], [joint1[1], joint2[1]], 'o-', lw=3, color='tab:green', label='link 2')
ax.plot([0, 0], [0, 1.25], '--', color='gray', lw=1.5, label='upright reference')

arc1 = Arc(pivot, 0.45, 0.45, angle=0, theta1=90 - np.degrees(theta1_demo), theta2=90, color='tab:blue', lw=2)
arc2 = Arc(joint1, 0.40, 0.40, angle=0, theta1=90, theta2=90 - np.degrees(theta2_demo), color='tab:green', lw=2)
ax.add_patch(arc1)
ax.add_patch(arc2)

ax.text(0.04, 0.28, r'$\theta_1$', color='tab:blue', fontsize=14)
ax.text(joint1[0] + 0.12, joint1[1] + 0.10, r'$\theta_2$', color='tab:green', fontsize=14)
ax.text(joint1[0] / 2 + 0.05, joint1[1] / 2, r'$l_1$', color='tab:blue', fontsize=13)
ax.text((joint1[0] + joint2[0]) / 2 + 0.04, (joint1[1] + joint2[1]) / 2, r'$l_2$', color='tab:green', fontsize=13)
ax.text(-0.08, -0.08, 'pivot', fontsize=11)

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-0.2, 2.0)
ax.set_aspect('equal', adjustable='box')
ax.set_title('Planar double inverted pendulum geometry')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper right')
plt.show()


## Problem Formulation

We use the state vector

$$
x = \begin{bmatrix} \theta_1 & \theta_2 & \dot{\theta}_1 & \dot{\theta}_2 \end{bmatrix}^T,
$$

where:

- $\theta_1$ is the angle of the first link from the upward vertical.
- $\theta_2$ is the angle of the second link from the upward vertical.
- $u$ is the torque applied at the first joint.

With masses $m_1, m_2$, lengths $l_1, l_2$, and gravity $g$, the nonlinear equations of motion are written here in first-order form:

$$
\dot{x} = f(x,u) =
\begin{bmatrix}
\dot{\theta}_1 \\
\dot{\theta}_2 \\
\ddot{\theta}_1 \\
\ddot{\theta}_2
\end{bmatrix},
$$

with $\Delta = \theta_1 - \theta_2$ and

$$
\ddot{\theta}_1 = \frac{g(2m_1+m_2)\sin\theta_1 + m_2 g \sin(\theta_1-2\theta_2) - 2m_2\sin\Delta\left(l_2\dot{\theta}_2^2 + l_1\dot{\theta}_1^2\cos\Delta\right) + 2u}{l_1\left(2m_1+m_2-m_2\cos(2\Delta)\right)},
$$

$$
\ddot{\theta}_2 = \frac{2\sin\Delta\left(l_1(m_1+m_2)\dot{\theta}_1^2 - g(m_1+m_2)\cos\theta_1 + l_2 m_2 \dot{\theta}_2^2\cos\Delta\right)}{l_2\left(2m_1+m_2-m_2\cos(2\Delta)\right)}.
$$

The upright equilibrium used for stabilization is

$$
x_e = \begin{bmatrix}0 & 0 & 0 & 0\end{bmatrix}^T,
\qquad u_e = 0.
$$

## Linearization Around the Upright Equilibrium

For small deviations $\delta x = x - x_e$ and $\delta u = u - u_e$, the system is approximated by

$$
\delta \dot{x} = A \, \delta x + B \, \delta u,
$$

with Jacobians evaluated at the upright equilibrium:

$$
A = \left.\frac{\partial f}{\partial x}\right|_{(x_e,u_e)}
= \begin{bmatrix}
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 \\
\frac{g(m_1+m_2)}{l_1 m_1} & -\frac{g m_2}{l_1 m_1} & 0 & 0 \\
-\frac{g(m_1+m_2)}{l_2 m_1} & \frac{g(m_1+m_2)}{l_2 m_1} & 0 & 0
\end{bmatrix},
$$

$$
B = \left.\frac{\partial f}{\partial u}\right|_{(x_e,u_e)}
= \begin{bmatrix}
0 \\
0 \\
\frac{1}{l_1 m_1} \\
0
\end{bmatrix}.
$$

The controllability of $(A,B)$ is checked numerically in the code below.

## LQR Objective

The continuous-time LQR controller minimizes the quadratic cost

$$
J = \int_0^{\infty} \left( \delta x^T Q \, \delta x + \delta u^T R \, \delta u \right) dt,
$$

where $Q \succeq 0$ penalizes state deviation and $R \succ 0$ penalizes control effort. The optimal feedback law is

$$
\delta u = -K \, \delta x,
$$

with

$$
K = R^{-1} B^T P,
$$

and $P$ the stabilizing solution of the continuous algebraic Riccati equation

$$
A^T P + P A - P B R^{-1} B^T P + Q = 0.
$$

The Riccati solution is computed directly from the Hamiltonian matrix using NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets

plt.rcParams['figure.figsize'] = (8, 4.5)

DEFAULT_PARAMS = {
    'm1': 1.0,
    'm2': 1.0,
    'l1': 1.0,
    'l2': 1.0,
    'g': 9.81,
}

X_EQ = np.zeros(4)
U_EQ = 0.0


In [ ]:
def dynamics(t, state, params, u=0.0):
    theta1, theta2, dtheta1, dtheta2 = state
    m1 = params['m1']
    m2 = params['m2']
    l1 = params['l1']
    l2 = params['l2']
    g = params['g']

    delta = theta1 - theta2
    denom = 2.0 * m1 + m2 - m2 * np.cos(2.0 * delta)

    ddtheta1 = (
        g * (2.0 * m1 + m2) * np.sin(theta1)
        + m2 * g * np.sin(theta1 - 2.0 * theta2)
        - 2.0 * np.sin(delta) * m2 * (dtheta2**2 * l2 + dtheta1**2 * l1 * np.cos(delta))
        + 2.0 * u
    ) / (l1 * denom)

    ddtheta2 = (
        2.0 * np.sin(delta) * (
            dtheta1**2 * l1 * (m1 + m2)
            - g * (m1 + m2) * np.cos(theta1)
            + dtheta2**2 * l2 * m2 * np.cos(delta)
        )
    ) / (l2 * denom)

    return np.array([dtheta1, dtheta2, ddtheta1, ddtheta2], dtype=float)


def linearized_matrices(params):
    m1 = params['m1']
    m2 = params['m2']
    l1 = params['l1']
    l2 = params['l2']
    g = params['g']

    A = np.array([
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
        [g * (m1 + m2) / (l1 * m1), -g * m2 / (l1 * m1), 0.0, 0.0],
        [-g * (m1 + m2) / (l2 * m1), g * (m1 + m2) / (l2 * m1), 0.0, 0.0],
    ])
    B = np.array([[0.0], [0.0], [1.0 / (l1 * m1)], [0.0]])
    return A, B


def controllability_matrix(A, B):
    n = A.shape[0]
    cols = [B]
    for k in range(1, n):
        cols.append(np.linalg.matrix_power(A, k) @ B)
    return np.hstack(cols)


def solve_care(A, B, Q, R):
    R_inv = np.linalg.inv(R)
    H = np.block([
        [A, -B @ R_inv @ B.T],
        [-Q, -A.T],
    ])

    eigvals, eigvecs = np.linalg.eig(H)
    stable = np.where(np.real(eigvals) < 0.0)[0]
    if len(stable) != A.shape[0]:
        raise RuntimeError('Failed to isolate the stable invariant subspace for the CARE.')

    V = eigvecs[:, stable]
    n = A.shape[0]
    U1 = V[:n, :]
    U2 = V[n:, :]
    P = np.real_if_close(U2 @ np.linalg.inv(U1), tol=1000)
    P = np.real(P)
    P = 0.5 * (P + P.T)
    return P.astype(float)


def lqr_gain(params, q_diag, r_scalar):
    A, B = linearized_matrices(params)
    Q = np.diag(q_diag)
    R = np.array([[r_scalar]], dtype=float)
    P = solve_care(A, B, Q, R)
    K = np.linalg.solve(R, B.T @ P)
    return K, P, A, B, Q, R


## Sources Used

- K. J. Astrom, K. Furuta, M. Iwashiro, T. Hoshino, *Energy Based Strategies for Swinging Up a Double Pendulum*, IFAC World Congress 1999.
- The local LQR stabilization workflow here uses the standard linearize-then-stabilize approach around the upright equilibrium.
- Practical reference project: [jitendra825/Inverted-Pendulum-Simulink](https://github.com/jitendra825/Inverted-Pendulum-Simulink).


In [ ]:
def rk4_step(f, t, x, dt):
    k1 = f(t, x)
    k2 = f(t + 0.5 * dt, x + 0.5 * dt * k1)
    k3 = f(t + 0.5 * dt, x + 0.5 * dt * k2)
    k4 = f(t + dt, x + dt * k3)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def make_lqr_controller(K):
    gain = np.asarray(K, dtype=float)
    def controller(t, x):
        return float(-(gain @ (np.asarray(x) - X_EQ))[0])
    return controller

def simulate(x0, params=None, duration=8.0, fps=40, controller=None, open_loop_torque=0.0, torque_limit=None):
    params = DEFAULT_PARAMS if params is None else params
    steps = max(2, int(duration * fps) + 1)
    t = np.linspace(0.0, duration, steps)
    x = np.zeros((4, steps), dtype=float)
    u_hist = np.zeros(steps, dtype=float)
    x[:, 0] = np.asarray(x0, dtype=float)

    def control_law(current_t, current_x):
        torque = float(open_loop_torque)
        if controller is not None:
            torque += float(controller(current_t, current_x))
        if torque_limit is not None:
            torque = float(np.clip(torque, -torque_limit, torque_limit))
        return torque

    for k in range(steps - 1):
        dt = t[k + 1] - t[k]
        def f(current_t, current_x):
            torque = control_law(current_t, current_x)
            return dynamics(current_t, current_x, params, u=torque)
        u_hist[k] = control_law(t[k], x[:, k])
        x[:, k + 1] = rk4_step(f, t[k], x[:, k], dt)
    u_hist[-1] = control_law(t[-1], x[:, -1])
    return {"t": t, "y": x, "u": u_hist}

def link_positions(sol, params=None):
    params = DEFAULT_PARAMS if params is None else params
    l1 = params["l1"]
    l2 = params["l2"]
    theta1 = sol["y"][0]
    theta2 = sol["y"][1]
    x1 = l1 * np.sin(theta1)
    y1 = l1 * np.cos(theta1)
    x2 = x1 + l2 * np.sin(theta2)
    y2 = y1 + l2 * np.cos(theta2)
    return x1, y1, x2, y2


## Controller Construction Check

This cell computes the controllability rank and the default LQR gain used in the notebook.

In [ ]:
A_default, B_default = linearized_matrices(DEFAULT_PARAMS)
ctrb = controllability_matrix(A_default, B_default)
rank_ctrb = np.linalg.matrix_rank(ctrb)
K_default, P_default, _, _, Q_default, R_default = lqr_gain(
    DEFAULT_PARAMS,
    q_diag=[120.0, 150.0, 18.0, 20.0],
    r_scalar=0.8,
)
display(Markdown(
    f"**Controllability rank:** {rank_ctrb} / {A_default.shape[0]}  \n"
    f"**Default LQR gain:** `K = {np.array2string(K_default, precision=3)}`"
))


In [ ]:
def plot_single_solution(sol, title):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(sol["t"], sol["y"][0], label=r"$\theta_1$")
    axes[0].plot(sol["t"], sol["y"][1], label=r"$\theta_2$")
    axes[0].set_title(f"{title}: angles")
    axes[0].set_xlabel("Time [s]")
    axes[0].set_ylabel("Angle [rad]")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[1].plot(sol["t"], sol["y"][2], label=r"$\dot{\theta}_1$")
    axes[1].plot(sol["t"], sol["y"][3], label=r"$\dot{\theta}_2$")
    axes[1].set_title(f"{title}: angular velocities")
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel("Angular velocity [rad/s]")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[2].plot(sol["t"], sol["u"], color="tab:red")
    axes[2].set_title(f"{title}: torque")
    axes[2].set_xlabel("Time [s]")
    axes[2].set_ylabel("Torque [Nm]")
    axes[2].grid(True, alpha=0.3)
    fig.tight_layout()
    return fig

def plot_comparison(open_loop_sol, lqr_sol):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(open_loop_sol["t"], open_loop_sol["y"][0], "--", label="free $\theta_1$")
    axes[0].plot(open_loop_sol["t"], open_loop_sol["y"][1], "--", label="free $\theta_2$")
    axes[0].plot(lqr_sol["t"], lqr_sol["y"][0], label="LQR $\theta_1$")
    axes[0].plot(lqr_sol["t"], lqr_sol["y"][1], label="LQR $\theta_2$")
    axes[0].set_title("Angles comparison")
    axes[0].set_xlabel("Time [s]")
    axes[0].set_ylabel("Angle [rad]")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=9)
    axes[1].plot(open_loop_sol["t"], np.linalg.norm(open_loop_sol["y"][:2], axis=0), "--", label="free")
    axes[1].plot(lqr_sol["t"], np.linalg.norm(lqr_sol["y"][:2], axis=0), label="LQR")
    axes[1].set_title("Distance from upright")
    axes[1].set_xlabel("Time [s]")
    axes[1].set_ylabel(r"$\|[\theta_1,\theta_2]\|_2$")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    axes[2].plot(open_loop_sol["t"], open_loop_sol["u"], "--", label="free torque")
    axes[2].plot(lqr_sol["t"], lqr_sol["u"], label="LQR torque")
    axes[2].set_title("Control effort comparison")
    axes[2].set_xlabel("Time [s]")
    axes[2].set_ylabel("Torque [Nm]")
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    fig.tight_layout()
    return fig

def animate_solution(sol, params=None, title="Simulation", interval=40, color="tab:blue", max_frames=180):
    params = DEFAULT_PARAMS if params is None else params
    x1, y1, x2, y2 = link_positions(sol, params)
    reach = params["l1"] + params["l2"]
    stride = max(1, len(sol["t"]) // max_frames)
    frame_idx = np.arange(0, len(sol["t"]), stride, dtype=int)
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.set_xlim(-reach - 0.2, reach + 0.2)
    ax.set_ylim(-reach - 0.2, reach + 0.2)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, alpha=0.25)
    ax.set_title(title)
    rod, = ax.plot([], [], "o-", lw=3, color=color)
    trail, = ax.plot([], [], "-", lw=1.5, alpha=0.45, color=color)
    time_text = ax.text(0.02, 0.95, "", transform=ax.transAxes)
    def update(frame_number):
        frame = frame_idx[frame_number]
        rod.set_data([0.0, x1[frame], x2[frame]], [0.0, y1[frame], y2[frame]])
        start = max(0, frame_number - 25)
        tail = frame_idx[start:frame_number + 1]
        trail.set_data(x2[tail], y2[tail])
        time_text.set_text(f"t = {sol["t"][frame]:.2f} s")
        return rod, trail, time_text
    ani = FuncAnimation(fig, update, frames=len(frame_idx), interval=interval, blit=True)
    html = HTML(ani.to_jshtml())
    plt.close(fig)
    return html


## Interactive Dashboard

Use `mode` to choose `Open-loop`, `LQR`, or `Compare`. This version is intentionally limited to the upright-equilibrium study.

In [ ]:
output = widgets.Output()
mode_dropdown = widgets.Dropdown(options=["Open-loop", "LQR", "Compare"], value="Compare", description="mode")
theta1_slider = widgets.FloatSlider(value=0.10, min=-0.5, max=0.5, step=0.01, description="theta1")
theta2_slider = widgets.FloatSlider(value=-0.08, min=-0.5, max=0.5, step=0.01, description="theta2")
dtheta1_slider = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.05, description="dtheta1")
dtheta2_slider = widgets.FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.05, description="dtheta2")
duration_slider = widgets.FloatSlider(value=8.0, min=2.0, max=20.0, step=0.5, description="duration")
torque_limit_slider = widgets.FloatSlider(value=12.0, min=1.0, max=30.0, step=0.5, description="limit")
q1_slider = widgets.FloatLogSlider(value=120.0, base=10, min=0.0, max=3.4, step=0.01, description="Q theta1")
q2_slider = widgets.FloatLogSlider(value=150.0, base=10, min=0.0, max=3.4, step=0.01, description="Q theta2")
q3_slider = widgets.FloatLogSlider(value=18.0, base=10, min=-1.0, max=3.0, step=0.01, description="Q dtheta1")
q4_slider = widgets.FloatLogSlider(value=20.0, base=10, min=-1.0, max=3.0, step=0.01, description="Q dtheta2")
r_slider = widgets.FloatLogSlider(value=0.8, base=10, min=-3.0, max=2.0, step=0.01, description="R")

def render(mode, theta1, theta2, dtheta1, dtheta2, duration, limit, q1, q2, q3, q4, r):
    with output:
        output.clear_output(wait=True)
        q_diag = [q1, q2, q3, q4]
        K, P, A, B, Q, R = lqr_gain(DEFAULT_PARAMS, q_diag=q_diag, r_scalar=r)
        lqr_controller = make_lqr_controller(K)
        x0 = [theta1, theta2, dtheta1, dtheta2]
        free_sol = simulate(x0, params=DEFAULT_PARAMS, duration=duration, fps=24, controller=None, torque_limit=limit)
        lqr_sol = simulate(x0, params=DEFAULT_PARAMS, duration=duration, fps=24, controller=lqr_controller, torque_limit=limit)
        eigvals = np.linalg.eigvals(A - B @ K)
        display(Markdown(
            f"**LQR gain** `K = {np.array2string(K, precision=3)}`  \n"
            f"**Closed-loop eigenvalues** `{np.array2string(eigvals, precision=3)}`"
        ))
        if mode == "Open-loop":
            fig = plot_single_solution(free_sol, "Open-loop")
            display(fig)
            plt.close(fig)
            display(animate_solution(free_sol, title="Open-loop animation", color="tab:orange"))
        elif mode == "LQR":
            fig = plot_single_solution(lqr_sol, "LQR closed-loop")
            display(fig)
            plt.close(fig)
            display(animate_solution(lqr_sol, title="LQR animation", color="tab:green"))
        else:
            fig = plot_comparison(free_sol, lqr_sol)
            display(fig)
            plt.close(fig)
            display(Markdown("**Open-loop animation**"))
            display(animate_solution(free_sol, title="Open-loop animation", color="tab:orange"))
            display(Markdown("**LQR animation**"))
            display(animate_solution(lqr_sol, title="LQR animation", color="tab:green"))

controls = {
    "mode": mode_dropdown,
    "theta1": theta1_slider,
    "theta2": theta2_slider,
    "dtheta1": dtheta1_slider,
    "dtheta2": dtheta2_slider,
    "duration": duration_slider,
    "limit": torque_limit_slider,
    "q1": q1_slider,
    "q2": q2_slider,
    "q3": q3_slider,
    "q4": q4_slider,
    "r": r_slider,
}
ui = widgets.VBox([
    widgets.HTML("<b>Upright-equilibrium study</b>"),
    mode_dropdown,
    theta1_slider,
    theta2_slider,
    dtheta1_slider,
    dtheta2_slider,
    duration_slider,
    torque_limit_slider,
    widgets.HTML("<b>LQR weights</b>"),
    q1_slider,
    q2_slider,
    q3_slider,
    q4_slider,
    r_slider,
])
interactive = widgets.interactive_output(render, controls)
display(ui, output, interactive)
